
# Forecast Resilience of ARIMA Models Under Macroeconomic Shocks

**Mentor-approved undergraduate research** testing whether standard ARIMA models lose forecast accuracy during macroeconomic volatility, and whether incorporating macroeconomic indicators improves resilience.

**Data:** 4 FRED time series (retail sales, CPI/inflation, federal funds rate, USD index), Jan 2019 - Dec 2024, split into stable (2019-20) / volatile (2021-22) / recovery (2023-24) periods.

## Key Findings
1. **Shock type mattered more than shock category** - the sudden COVID shock (sitting inside the "stable" 2019-20 period) degraded forecast accuracy far more than the gradual 2021-22 inflationary volatility the framework was built around.
2. **Macro-awareness helps, but only when applied selectively** - one well-chosen macroeconomic variable improved forecast resilience; adding all three at once did not.
3. **Model complexity has real limits given data availability** - a fully-specified VAR became infeasible on this dataset, supporting simpler, selective models when historical data is limited.



1. DATA LOADING AND CLEANING

Cleaning and merging of all 4 datasets.
Goal: one clean table, monthly, Jan 2019-Dec 2024, with:
RSAFS (Retail sales dataset) : 'retail_sales' (feature to be forecasted)
CPIAUCSL (Inflation Data): 'inflation_yoy' (added feature for crosschecking prediction accuracies)
FEDFUNDS (interest rates) : 'interest_rate'
TWEXBGSMTH (Dollar index ) :'dollar_index'



In [2]:
import pandas as pd 

STEP 1 : LOAD EACH FILE 

# STEP 1: Load each file
Each FRED file has 2 columns: 'observation_date' and the value.
We tell pandas to treat 'observation_date' as an actual date,
not just text, so we can filter and merge by date later.



In [3]:
rsafs = pd.read_csv(r"data/RSAFS.csv", parse_dates=["observation_date"],dayfirst=True)

cpi = pd.read_csv(r"data/CPIAUCSL.csv", parse_dates=["observation_date"],dayfirst=True) 

fedfunds = pd.read_csv(r"data/FEDFUNDS.csv", parse_dates=["observation_date"],dayfirst=True)

dollar = pd.read_csv(r"data/TWEXBGSMTH.csv", parse_dates=["observation_date"])

STEP 2 : RENAME COLUMNS TO CLEAR , HUMAN - READABLE NAMES 

In [4]:
rsafs = rsafs.rename(columns={"observation_date": "date", "RSAFS": "retail_sales"})

cpi = cpi.rename(columns={"observation_date": "date", "CPIAUCSL": "cpi_index"})

fedfunds = fedfunds.rename(columns={"observation_date": "date", "FEDFUNDS": "interest_rate"})

dollar = dollar.rename(columns={"observation_date": "date", "TWEXBGSMTH": "dollar_index"})

STEP 3: Turn raw CPI into "inflation" (year-over-year % change)
CPI on its own (e.g. "252.5") means nothing to a normal reader.
What matters is: how much did prices rise compared to 12 months ago?
pct_change(12) does exactly that on monthly data (12 months back).

In [5]:
cpi = cpi.sort_values("date")
cpi["inflation_yoy"] = cpi["cpi_index"].pct_change(12) * 100
cpi = cpi[["date", "inflation_yoy"]]   #since we only need the transformed column now 

In [6]:
print(rsafs.shape)
print(rsafs.head())
print(rsafs.tail())
print(rsafs['date'].dt.day.unique())
print(rsafs['date'].duplicated().sum())
print(rsafs.isnull().sum())

(72, 2)
        date  retail_sales
0 2019-01-01        490440
1 2019-02-01        491751
2 2019-03-01        499292
3 2019-04-01        499343
4 2019-05-01        504741
         date  retail_sales
67 2024-08-01        697039
68 2024-09-01        703055
69 2024-10-01        708290
70 2024-11-01        711853
71 2024-12-01        717265
[1]
0
date            0
retail_sales    0
dtype: int64


In [7]:
print(cpi.shape)
print(cpi.head())
print(cpi.tail())
print(cpi['date'].dt.day.unique())
print(cpi['date'].duplicated().sum())
print(cpi.isnull().sum())

(84, 2)
        date  inflation_yoy
0 2018-01-01            NaN
1 2018-02-01            NaN
2 2018-03-01            NaN
3 2018-04-01            NaN
4 2018-05-01            NaN
         date  inflation_yoy
79 2024-08-01       2.607144
80 2024-09-01       2.426483
81 2024-10-01       2.578844
82 2024-11-01       2.719472
83 2024-12-01       2.870691
[1]
0
date              0
inflation_yoy    12
dtype: int64


In [8]:
print(fedfunds.shape)
print(fedfunds.head())
print(fedfunds.tail())
print(fedfunds['date'].dt.day.unique())
print(fedfunds['date'].duplicated().sum())
print(fedfunds.isnull().sum())

(72, 2)
        date  interest_rate
0 2019-01-01           2.40
1 2019-02-01           2.40
2 2019-03-01           2.41
3 2019-04-01           2.42
4 2019-05-01           2.39
         date  interest_rate
67 2024-08-01           5.33
68 2024-09-01           5.13
69 2024-10-01           4.83
70 2024-11-01           4.64
71 2024-12-01           4.48
[1]
0
date             0
interest_rate    0
dtype: int64


In [9]:
print(dollar.shape)
print(dollar.head())
print(dollar.tail())
print(dollar['date'].dt.day.unique())
print(dollar['date'].duplicated().sum())
print(dollar.isnull().sum())

(72, 2)
        date  dollar_index
0 2019-01-01      114.4425
1 2019-02-01      114.4130
2 2019-03-01      114.7835
3 2019-04-01      114.9037
4 2019-05-01      115.9574
         date  dollar_index
67 2024-08-01      122.6137
68 2024-09-01      121.8849
69 2024-10-01      123.5953
70 2024-11-01      126.3113
71 2024-12-01      127.5758
[1]
0
date            0
dollar_index    0
dtype: int64



STEP 4: Merge everything into a single table, matched by date
"outer" merge keeps every date from every file, even if one file
is missing that date (like FEDFUNDS after Jan 2024) - this lets us
SEE the gap clearly instead of silently losing data.


In [10]:
merged = rsafs.merge(cpi, on="date", how="outer")
merged = merged.merge(fedfunds, on="date", how="outer")
merged = merged.merge(dollar, on="date", how="outer")
merged = merged.sort_values("date").reset_index(drop=True)


STEP 5: Keep only our research window: Jan 2019 - Dec 2024
(CPI file has extra years beyond 2024 that we don't need)


In [11]:
merged = merged[(merged["date"] >= "2019-01-01") & (merged["date"] <= "2024-12-31")]
merged = merged.reset_index(drop=True)

In [12]:
print(merged['date'].min())
print(merged['date'].max())

2019-01-01 00:00:00
2024-12-01 00:00:00


In [13]:
print(merged.shape)

print("RSAFS:", rsafs.shape)
print("CPI:", cpi.shape)
print("FEDFUNDS:", fedfunds.shape)
print("Dollar:", dollar.shape)

(72, 5)
RSAFS: (72, 2)
CPI: (84, 2)
FEDFUNDS: (72, 2)
Dollar: (72, 2)



STEP 6: Tag each row with its research sub-period


In [14]:
def tag_period(date):
    if date <= pd.Timestamp("2020-12-31"):
        return "stable"
    elif date <= pd.Timestamp("2022-12-31"):
        return "volatile"
    else:
        return "recovery"

merged["period"] = merged["date"].apply(tag_period)


STEP 7: Report data quality - show any missing values clearly


In [15]:
print("=== Shape of final merged table ===")
print(merged.shape, "\n")

print("=== Missing values per column ===")
print(merged.isna().sum(), "\n")

print("=== Rows with missing interest_rate (the FEDFUNDS gap) ===")
print(merged[merged["interest_rate"].isna()][["date", "interest_rate"]], "\n")

print("=== Row count per period ===")
print(merged["period"].value_counts(), "\n")

print("=== First 5 rows ===")
print(merged.head(), "\n")

print("=== Last 5 rows ===")
print(merged.tail(), "\n")

=== Shape of final merged table ===
(72, 6) 

=== Missing values per column ===
date             0
retail_sales     0
inflation_yoy    0
interest_rate    0
dollar_index     0
period           0
dtype: int64 

=== Rows with missing interest_rate (the FEDFUNDS gap) ===
Empty DataFrame
Columns: [date, interest_rate]
Index: [] 

=== Row count per period ===
period
stable      24
volatile    24
recovery    24
Name: count, dtype: int64 

=== First 5 rows ===
        date  retail_sales  inflation_yoy  interest_rate  dollar_index  period
0 2019-01-01      490440.0       1.487589           2.40      114.4425  stable
1 2019-02-01      491751.0       1.518862           2.40      114.4130  stable
2 2019-03-01      499292.0       1.883186           2.41      114.7835  stable
3 2019-04-01      499343.0       2.000583           2.42      114.9037  stable
4 2019-05-01      504741.0       1.795911           2.39      115.9574  stable 

=== Last 5 rows ===
         date  retail_sales  inflation_yoy  int


**Data-quality investigation:** the merge above showed missing `interest_rate` values in recent months. Before assuming this was a data problem, we went back to inspect the raw source files directly rather than trusting a first-pass `isnull()` check.


In [16]:
import pandas as pd

# Load each file individually and inspect BEFORE merging
fedfunds_check = pd.read_csv(r"data/FEDFUNDS.csv", parse_dates=["observation_date"])
print("FEDFUNDS shape:", fedfunds_check.shape)
print(fedfunds_check.head(10))
print(fedfunds_check.tail(5))

cpi_check = pd.read_csv(r"data/CPIAUCSL.csv", parse_dates=["observation_date"])
print("\nCPI shape:", cpi_check.shape)
print(cpi_check.head(10))

FEDFUNDS shape: (72, 2)
  observation_date  FEDFUNDS
0       2019-01-01      2.40
1       2019-01-02      2.40
2       2019-01-03      2.41
3       2019-01-04      2.42
4       2019-01-05      2.39
5       2019-01-06      2.38
6       2019-01-07      2.40
7       2019-01-08      2.13
8       2019-01-09      2.04
9       2019-01-10      1.83
   observation_date  FEDFUNDS
67       2024-01-08      5.33
68       2024-01-09      5.13
69       2024-01-10      4.83
70       2024-01-11      4.64
71       2024-01-12      4.48

CPI shape: (84, 2)
  observation_date  CPIAUCSL
0       2018-01-01   248.859
1       2018-01-02   249.529
2       2018-01-03   249.577
3       2018-01-04   250.227
4       2018-01-05   250.792
5       2018-01-06   251.018
6       2018-01-07   251.214
7       2018-01-08   251.663
8       2018-01-09   252.182
9       2018-01-10   252.772


In [24]:
print(merged.shape)
print(merged.isna().sum())
print(merged['period'].value_counts())

(72, 6)
date             0
retail_sales     0
inflation_yoy    0
interest_rate    0
dollar_index     0
period           0
dtype: int64
period
stable      24
volatile    24
recovery    24
Name: count, dtype: int64


In [25]:
print("RSAFS:", rsafs.shape, rsafs['date'].dt.day.unique(), rsafs['date'].duplicated().sum())
print("CPI:", cpi.shape, cpi['date'].dt.day.unique(), cpi['date'].duplicated().sum())
print("FEDFUNDS:", fedfunds.shape, fedfunds['date'].dt.day.unique(), fedfunds['date'].duplicated().sum())
print("Dollar:", dollar.shape, dollar['date'].dt.day.unique(), dollar['date'].duplicated().sum())
print()
print("MERGED:", merged.shape)
print(merged.isna().sum())

RSAFS: (72, 2) [1] 0
CPI: (84, 2) [1] 0
FEDFUNDS: (72, 2) [1] 0
Dollar: (72, 2) [1] 0

MERGED: (72, 6)
date             0
retail_sales     0
inflation_yoy    0
interest_rate    0
dollar_index     0
period           0
dtype: int64


In [26]:
with open(r"data/TWEXBGSMTH.csv", "r") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i > 15:
            break

observation_date,TWEXBGSMTH
2019-01-01,114.4425
2019-02-01,114.4130
2019-03-01,114.7835
2019-04-01,114.9037
2019-05-01,115.9574
2019-06-01,115.4271
2019-07-01,115.0776
2019-08-01,117.1004
2019-09-01,117.3239
2019-10-01,116.7463
2019-11-01,116.5595
2019-12-01,115.9096
2020-01-01,115.2794
2020-02-01,116.7105
2020-03-01,121.0227
2020-04-01,123.2803


In [28]:
print(merged.shape)
print(merged.isna().sum())
print(merged['period'].value_counts())


(72, 6)
date             0
retail_sales     0
inflation_yoy    0
interest_rate    0
dollar_index     0
period           0
dtype: int64
period
stable      24
volatile    24
recovery    24
Name: count, dtype: int64



## 2. Stationarity Testing (ADF)


In [29]:
from statsmodels.tsa.stattools import adfuller

df = merged

print("=== ADF Test for all three periods ===")
for period_name in ["stable", "volatile", "recovery"]:
    period_data = df[df["period"] == period_name].sort_values("date")["retail_sales"]
    result = adfuller(period_data)
    verdict = "d = 0 (stationary)" if result[1] < 0.05 else "d = 1 (non-stationary, has trend)"
    print(f"{period_name:10s} -> ADF stat = {result[0]:.4f}, p-value = {result[1]:.4f} -> {verdict}")

=== ADF Test for all three periods ===
stable     -> ADF stat = -2.7271, p-value = 0.0695 -> d = 1 (non-stationary, has trend)
volatile   -> ADF stat = -2.2661, p-value = 0.1831 -> d = 1 (non-stationary, has trend)
recovery   -> ADF stat = 4.1551, p-value = 1.0000 -> d = 1 (non-stationary, has trend)


C:\Users\HP\AppData\Local\Temp\ipykernel_24048\404628531.py:8: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  result = adfuller(period_data)
C:\Users\HP\AppData\Local\Temp\ipykernel_24048\404628531.py:8: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  result = adfuller(period_data)
C:\Users\HP\AppData\Local\Temp\ipykernel_24048\404628531.py:8: FutureWarning: adfuller current


## 3. Baseline ARIMA Modeling


In [30]:
!pip install pmdarima
!pip install --upgrade statsmodels pmdarima

In [31]:
import pmdarima as pm

print("=== Auto ARIMA results for all three periods ===\n")

for period_name in ["stable", "volatile", "recovery"]:
    period_data = df[df["period"] == period_name].sort_values("date")["retail_sales"]
    
    model = pm.auto_arima(
        period_data,
        d=1,                  # we already know d=1 from the ADF test
        seasonal=False,       # no seasonal component for now (24 months is short for seasonality)
        trace=True,           # prints out every combination it tries
        suppress_warnings=True,
        stepwise=True         # smart search instead of testing every possible combo
    )
    
    print(f"\n{period_name.upper()} — Best model: {model.order}")
    print(model.summary())
    print("\n" + "="*80 + "\n")

=== Auto ARIMA results for all three periods ===

Performing stepwise search to minimize aic
 ARIMA(2,1,2)(0,0,0)[0] intercept   : AIC=543.287, Time=0.74 sec
 ARIMA(0,1,0)(0,0,0)[0] intercept   : AIC=535.445, Time=0.03 sec
 ARIMA(1,1,0)(0,0,0)[0] intercept   : AIC=537.455, Time=0.05 sec
 ARIMA(0,1,1)(0,0,0)[0] intercept   : AIC=537.782, Time=0.06 sec
 ARIMA(0,1,0)(0,0,0)[0]             : AIC=533.624, Time=0.04 sec
 ARIMA(1,1,1)(0,0,0)[0] intercept   : AIC=539.782, Time=0.08 sec

Best model:  ARIMA(0,1,0)(0,0,0)[0]          
Total fit time: 1.010 seconds

STABLE — Best model: (0, 1, 0)
                               SARIMAX Results                                
Dep. Variable:                      y   No. Observations:                   24
Model:               SARIMAX(0, 1, 0)   Log Likelihood                -265.812
Date:                Tue, 15 Sep 2026   AIC                            533.624
Time:                        14:01:42   BIC                            534.760
Sample:      

In [32]:
import numpy as np
from statsmodels.tsa.arima.model import ARIMA

# Best (p,d,q) orders found using auto_arima for each period
best_orders = {
    "stable": (0, 1, 0),
    "volatile": (0, 1, 2),
    "recovery": (0, 1, 2),
}

INITIAL_TRAIN_SIZE = 12  # months of history before we start testing

results_summary = []

for period_name in ["stable", "volatile", "recovery"]:
    print(f"\n{'='*60}")
    print(f"PERIOD: {period_name.upper()}")
    print(f"{'='*60}")

    period_data = df[df["period"] == period_name].sort_values("date")["retail_sales"].reset_index(drop=True)
    order = best_orders[period_name]
    actuals, predictions = [], []

    for i in range(INITIAL_TRAIN_SIZE, len(period_data)):
        train = period_data.iloc[:i]
        actual_value = period_data.iloc[i]

        model = ARIMA(train, order=order)
        fitted_model = model.fit()
        forecast = fitted_model.forecast(steps=1)
        predicted_value = forecast.iloc[0]

        actuals.append(actual_value)
        predictions.append(predicted_value)
        print(f"Month {i+1:2d}: actual = {actual_value:>10,.0f}   predicted = {predicted_value:>10,.0f}")

    actuals = np.array(actuals)
    predictions = np.array(predictions)
    rmse = np.sqrt(np.mean((actuals - predictions) ** 2))
    mape = np.mean(np.abs((actuals - predictions) / actuals)) * 100

    print(f"\n{period_name.upper()} RESULTS: RMSE = {rmse:,.2f}   MAPE = {mape:.2f}%")
    results_summary.append({"period": period_name, "order": order, "rmse": rmse, "mape": mape, "n_predictions": len(actuals)})

print(f"\n{'='*60}\nSUMMARY TABLE\n{'='*60}")
summary_df = pd.DataFrame(results_summary)
print(summary_df.to_string(index=False))


PERIOD: STABLE
Month 13: actual =    515,119   predicted =    515,866
Month 14: actual =    515,330   predicted =    515,119
Month 15: actual =    468,324   predicted =    515,330
Month 16: actual =    401,028   predicted =    468,324
Month 17: actual =    478,449   predicted =    401,028
Month 18: actual =    518,038   predicted =    478,449
Month 19: actual =    526,304   predicted =    518,038
Month 20: actual =    530,735   predicted =    526,304
Month 21: actual =    541,302   predicted =    530,735
Month 22: actual =    539,114   predicted =    541,302
Month 23: actual =    533,941   predicted =    539,114
Month 24: actual =    538,548   predicted =    533,941

STABLE RESULTS: RMSE = 34,824.10   MAPE = 4.78%

PERIOD: VOLATILE


C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params


Month 13: actual =    631,509   predicted =    620,748
Month 14: actual =    638,101   predicted =    630,615
Month 15: actual =    651,027   predicted =    639,089
Month 16: actual =    660,194   predicted =    651,551
Month 17: actual =    659,847   predicted =    661,290
Month 18: actual =    666,113   predicted =    660,830
Month 19: actual =    659,550   predicted =    665,826
Month 20: actual =    663,566   predicted =    660,284
Month 21: actual =    661,854   predicted =    662,794
Month 22: actual =    668,671   predicted =    662,239
Month 23: actual =    659,458   predicted =    668,409
Month 24: actual =    651,763   predicted =    660,388

VOLATILE RESULTS: RMSE = 7,449.57   MAPE = 1.02%

PERIOD: RECOVERY
Month 13: actual =    680,456   predicted =    686,274


C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


Month 14: actual =    685,280   predicted =    680,521
Month 15: actual =    687,641   predicted =    685,283
Month 16: actual =    687,602   predicted =    687,557
Month 17: actual =    692,774   predicted =    687,575
Month 18: actual =    692,449   predicted =    692,707
Month 19: actual =    698,835   predicted =    692,395
Month 20: actual =    697,039   predicted =    698,756
Month 21: actual =    703,055   predicted =    696,991
Month 22: actual =    708,290   predicted =    702,995
Month 23: actual =    711,853   predicted =    708,158
Month 24: actual =    717,265   predicted =    711,750

RECOVERY RESULTS: RMSE = 4,495.04   MAPE = 0.56%

SUMMARY TABLE
  period     order         rmse     mape  n_predictions
  stable (0, 1, 0) 34824.096150 4.784617             12
volatile (0, 1, 2)  7449.565220 1.021872             12
recovery (0, 1, 2)  4495.041606 0.562714             12



## 5. Shock-Type Analysis: COVID Crash vs. Labeled Volatility

The "stable" period (2019-2020) technically includes the COVID crash months (March-June 2020). Re-running walk-forward validation while explicitly excluding those months isolates how much of the "stable" period's error actually came from the COVID shock itself, versus ordinary noise.


In [34]:
covid_crash_dates = pd.to_datetime(["2020-03-01", "2020-04-01", "2020-05-01", "2020-06-01"])

stable_data = df[df["period"] == "stable"].sort_values("date").reset_index(drop=True)
order = best_orders["stable"]

actuals_excl, predictions_excl = [], []

for i in range(INITIAL_TRAIN_SIZE, len(stable_data)):
    train = stable_data["retail_sales"].iloc[:i]
    actual_value = stable_data["retail_sales"].iloc[i]
    actual_date = stable_data["date"].iloc[i]

    model = ARIMA(train, order=order)
    fitted_model = model.fit()
    forecast = fitted_model.forecast(steps=1)
    predicted_value = forecast.iloc[0]

    if actual_date in covid_crash_dates:
        print(f"{actual_date.date()}: EXCLUDED (COVID crash month)")
        continue

    actuals_excl.append(actual_value)
    predictions_excl.append(predicted_value)

actuals_excl = np.array(actuals_excl)
predictions_excl = np.array(predictions_excl)
rmse_excl = np.sqrt(np.mean((actuals_excl - predictions_excl) ** 2))
mape_excl = np.mean(np.abs((actuals_excl - predictions_excl) / actuals_excl)) * 100

print(f"\nStable period EXCLUDING COVID months: RMSE = {rmse_excl:,.2f}   MAPE = {mape_excl:.2f}%")
print(f"Stable period INCLUDING COVID months: RMSE = 34,824.10   MAPE = 4.78%")


2020-03-01: EXCLUDED (COVID crash month)
2020-04-01: EXCLUDED (COVID crash month)
2020-05-01: EXCLUDED (COVID crash month)
2020-06-01: EXCLUDED (COVID crash month)

Stable period EXCLUDING COVID months: RMSE = 5,623.57   MAPE = 0.85%
Stable period INCLUDING COVID months: RMSE = 34,824.10   MAPE = 4.78%



## 6. ARIMAX Extension: Incorporating Macroeconomic Indicators

Does adding macro context (inflation, interest rates, the dollar index) improve forecast resilience? Tested three ways: all three variables together, one variable at a time, and inflation alone.


In [35]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA

exog_cols = ["inflation_yoy", "interest_rate", "dollar_index"]

# Best (p,d,q) orders found via auto_arima WITH exogenous variables
best_orders_arimax = {
    "stable": (0, 1, 0),
    "volatile": (1, 1, 3),
    "recovery": (2, 1, 0),
}

# Baseline results, already computed, for direct comparison
baseline_results = {
    "stable": {"rmse": 34824.10, "mape": 4.78},
    "volatile": {"rmse": 7449.57, "mape": 1.02},
    "recovery": {"rmse": 4495.04, "mape": 0.56},
}

INITIAL_TRAIN_SIZE = 12

arimax_results = []

for period_name in ["stable", "volatile", "recovery"]:
    print(f"\n{'='*60}")
    print(f"PERIOD: {period_name.upper()} (ARIMAX)")
    print(f"{'='*60}")

    period_df = df[df["period"] == period_name].sort_values("date").reset_index(drop=True)
    y = period_df["retail_sales"]
    X = period_df[exog_cols]

    order = best_orders_arimax[period_name]
    actuals, predictions = [], []

    for i in range(INITIAL_TRAIN_SIZE, len(period_df)):
        y_train = y.iloc[:i]
        X_train = X.iloc[:i]
        X_next = X.iloc[i:i+1]
        actual_value = y.iloc[i]

        model = ARIMA(y_train, order=order, exog=X_train)
        fitted_model = model.fit()

        forecast = fitted_model.forecast(steps=1, exog=X_next)
        predicted_value = forecast.iloc[0]

        actuals.append(actual_value)
        predictions.append(predicted_value)
        print(f"Month {i+1:2d}: actual = {actual_value:>10,.0f}   predicted = {predicted_value:>10,.0f}")

    actuals = np.array(actuals)
    predictions = np.array(predictions)
    rmse = np.sqrt(np.mean((actuals - predictions) ** 2))
    mape = np.mean(np.abs((actuals - predictions) / actuals)) * 100

    print(f"\n{period_name.upper()} ARIMAX RESULTS: RMSE = {rmse:,.2f}   MAPE = {mape:.2f}%")

    arimax_results.append({
        "period": period_name, "order": order, "rmse": rmse, "mape": mape, "n_predictions": len(actuals)
    })

print(f"\n{'='*70}")
print("COMPARISON: BASELINE (ARIMA) vs ENHANCED (ARIMAX)")
print(f"{'='*70}")

comparison_rows = []
for r in arimax_results:
    p = r["period"]
    base_mape = baseline_results[p]["mape"]
    enh_mape = r["mape"]
    improvement = base_mape - enh_mape
    comparison_rows.append({
        "period": p, "baseline_mape": base_mape, "arimax_mape": round(enh_mape, 2),
        "improvement_pct_points": round(improvement, 2),
        "did_it_help": "YES" if improvement > 0 else "NO"
    })

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))
comparison_df.to_csv("baseline_vs_arimax_comparison.csv", index=False)


PERIOD: STABLE (ARIMAX)
Month 13: actual =    515,119   predicted =    517,623
Month 14: actual =    515,330   predicted =    516,005
Month 15: actual =    468,324   predicted =    517,928
Month 16: actual =    401,028   predicted =    425,420
Month 17: actual =    478,449   predicted =    392,960
Month 18: actual =    518,038   predicted =    499,360
Month 19: actual =    526,304   predicted =    530,149
Month 20: actual =    530,735   predicted =    542,667
Month 21: actual =    541,302   predicted =    535,870
Month 22: actual =    539,114   predicted =    540,618
Month 23: actual =    533,941   predicted =    546,140
Month 24: actual =    538,548   predicted =    545,648

STABLE ARIMAX RESULTS: RMSE = 30,424.15   MAPE = 3.89%

PERIOD: VOLATILE (ARIMAX)


C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params


Month 13: actual =    631,509   predicted =    612,690


C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params


Month 14: actual =    638,101   predicted =    638,001


...

Month 22: actual =    668,671   predicted =    660,106


C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params


Month 23: actual =    659,458   predicted =    657,539


C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params


Month 24: actual =    651,763   predicted =    648,470

VOLATILE ARIMAX RESULTS: RMSE = 21,109.29   MAPE = 2.31%

PERIOD: RECOVERY (ARIMAX)
Month 13: actual =    680,456   predicted =    685,526
Month 14: actual =    685,280   predicted =    680,637
Month 15: actual =    687,641   predicted =    686,441
Month 16: actual =    687,602   predicted =    687,126
Month 17: actual =    692,774   predicted =    687,141
Month 18: actual =    692,449   predicted =    691,639
Month 19: actual =    698,835   predicted =    692,296
Month 20: actual =    697,039   predicted =    697,650
Month 21: actual =    703,055   predicted =    694,700
Month 22: actual =    708,290   predicted =    702,817
Month 23: actual =    711,853   predicted =    709,596
Month 24: actual =    717,265   predicted =    713,294

RECOVERY ARIMAX RESULTS: RMSE = 4,519.24   MAPE = 0.54%

COMPARISON: BASELINE (ARIMA) vs ENHANCED (ARIMAX)
  period  baseline_mape  arimax_mape  improvement_pct_points did_it_help
  stable           

In [36]:
print(f"\n{'='*70}")
print("SINGLE-VARIABLE TEST: VOLATILE PERIOD")
print(f"{'='*70}")

single_var_results = []

volatile_df = df[df["period"] == "volatile"].sort_values("date").reset_index(drop=True)
y = volatile_df["retail_sales"]

for var in ["inflation_yoy", "interest_rate", "dollar_index"]:
    print(f"\n--- Testing with only: {var} ---")

    X = volatile_df[[var]]

    # Re-find best order for this single-variable version
    order = pm.auto_arima(y, X=X, d=1, seasonal=False, trace=False, suppress_warnings=True, stepwise=True).order
    print(f"Best order for {var}: {order}")

    actuals, predictions = [], []

    for i in range(INITIAL_TRAIN_SIZE, len(volatile_df)):
        y_train = y.iloc[:i]
        X_train = X.iloc[:i]
        X_next = X.iloc[i:i+1]
        actual_value = y.iloc[i]

        model = ARIMA(y_train, order=order, exog=X_train)
        fitted_model = model.fit()
        forecast = fitted_model.forecast(steps=1, exog=X_next)
        predicted_value = forecast.iloc[0]

        actuals.append(actual_value)
        predictions.append(predicted_value)

    actuals = np.array(actuals)
    predictions = np.array(predictions)
    rmse = np.sqrt(np.mean((actuals - predictions) ** 2))
    mape = np.mean(np.abs((actuals - predictions) / actuals)) * 100

    print(f"{var} ALONE: RMSE = {rmse:,.2f}   MAPE = {mape:.2f}%")
    single_var_results.append({"variable": var, "order": order, "rmse": rmse, "mape": mape})

print(f"\n{'='*70}")
print("SUMMARY: SINGLE-VARIABLE COMPARISON (VOLATILE PERIOD)")
print(f"{'='*70}")
single_var_df = pd.DataFrame(single_var_results)
print(single_var_df.to_string(index=False))
print(f"\nFor reference:")
print(f"  Baseline (no macro vars):        MAPE = 1.02%")
print(f"  All 3 macro vars together:       MAPE = 2.04%")

single_var_df.to_csv("volatile_single_variable_test.csv", index=False)


SINGLE-VARIABLE TEST: VOLATILE PERIOD

--- Testing with only: inflation_yoy ---
Best order for inflation_yoy: (0, 1, 4)


C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:989: EstimationWarning: Too few observations to estimate starting parameters for ARMA and trend. All parameters except for variances will be set to zeros.
  params_variance) = self._conditional_sum_squares(
C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:989: EstimationWarning: Too few observations to estimate starting parameters for ARMA and trend. All parameters except for variances will be set to zeros.
  params_variance) = self._conditional_sum_squares(
C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start

inflation_yoy ALONE: RMSE = 6,291.10   MAPE = 0.68%

--- Testing with only: interest_rate ---
Best order for interest_rate: (0, 1, 2)
interest_rate ALONE: RMSE = 21,397.60   MAPE = 1.97%

--- Testing with only: dollar_index ---
Best order for dollar_index: (0, 1, 2)


C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params


dollar_index ALONE: RMSE = 10,729.94   MAPE = 1.35%

SUMMARY: SINGLE-VARIABLE COMPARISON (VOLATILE PERIOD)
     variable     order         rmse     mape
inflation_yoy (0, 1, 4)  6291.098637 0.682921
interest_rate (0, 1, 2) 21397.595804 1.968956
 dollar_index (0, 1, 2) 10729.935269 1.347925

For reference:
  Baseline (no macro vars):        MAPE = 1.02%
  All 3 macro vars together:       MAPE = 2.04%


C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


In [37]:
print(f"\n{'='*70}")
print("INFLATION-ONLY MODEL: ALL THREE PERIODS")
print(f"{'='*70}")

inflation_results = []

for period_name in ["stable", "volatile", "recovery"]:
    print(f"\n--- {period_name.upper()} (inflation_yoy only) ---")

    period_df = df[df["period"] == period_name].sort_values("date").reset_index(drop=True)
    y = period_df["retail_sales"]
    X = period_df[["inflation_yoy"]]

    # Find best order for this period using only inflation as the exogenous variable
    order = pm.auto_arima(y, X=X, d=1, seasonal=False, trace=False, suppress_warnings=True, stepwise=True).order
    print(f"Best order: {order}")

    actuals, predictions = [], []

    for i in range(INITIAL_TRAIN_SIZE, len(period_df)):
        y_train = y.iloc[:i]
        X_train = X.iloc[:i]
        X_next = X.iloc[i:i+1]
        actual_value = y.iloc[i]

        model = ARIMA(y_train, order=order, exog=X_train)
        fitted_model = model.fit()
        forecast = fitted_model.forecast(steps=1, exog=X_next)
        predicted_value = forecast.iloc[0]

        actuals.append(actual_value)
        predictions.append(predicted_value)

    actuals = np.array(actuals)
    predictions = np.array(predictions)
    rmse = np.sqrt(np.mean((actuals - predictions) ** 2))
    mape = np.mean(np.abs((actuals - predictions) / actuals)) * 100

    print(f"{period_name.upper()} (inflation only): RMSE = {rmse:,.2f}   MAPE = {mape:.2f}%")
    inflation_results.append({"period": period_name, "order": order, "rmse": rmse, "mape": mape})

# -----------------------------------------------------------------
# Final combined comparison: Baseline vs All-3-macro-vars vs Inflation-only
# -----------------------------------------------------------------
print(f"\n{'='*70}")
print("FINAL COMPARISON: BASELINE vs ARIMAX (all 3 vars) vs INFLATION-ONLY")
print(f"{'='*70}")

# Fill in your actual baseline and all-3-var MAPE values from earlier results
baseline_mape = {"stable": 4.78, "volatile": 1.02, "recovery": 0.56}
all3_mape = {"stable": 3.89, "volatile": 2.31, "recovery": 0.54}

final_comparison = []
for r in inflation_results:
    p = r["period"]
    final_comparison.append({
        "period": p,
        "baseline_mape": baseline_mape[p],
        "all_3_vars_mape": all3_mape[p],
        "inflation_only_mape": round(r["mape"], 2),
    })

final_df = pd.DataFrame(final_comparison)
print(final_df.to_string(index=False))

final_df.to_csv("final_model_comparison.csv", index=False)
print("\nSaved final_model_comparison.csv")


INFLATION-ONLY MODEL: ALL THREE PERIODS

--- STABLE (inflation_yoy only) ---
Best order: (0, 1, 0)
STABLE (inflation only): RMSE = 28,598.16   MAPE = 3.56%

--- VOLATILE (inflation_yoy only) ---
Best order: (0, 1, 4)


C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:989: EstimationWarning: Too few observations to estimate starting parameters for ARMA and trend. All parameters except for variances will be set to zeros.
  params_variance) = self._conditional_sum_squares(
C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:989: EstimationWarning: Too few observations to estimate starting parameters for ARMA and trend. All parameters except for variances will be set to zeros.
  params_variance) = self._conditional_sum_squares(
C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start

VOLATILE (inflation only): RMSE = 6,291.10   MAPE = 0.68%

--- RECOVERY (inflation_yoy only) ---
Best order: (2, 1, 0)


C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


RECOVERY (inflation only): RMSE = 4,357.20   MAPE = 0.53%

FINAL COMPARISON: BASELINE vs ARIMAX (all 3 vars) vs INFLATION-ONLY
  period  baseline_mape  all_3_vars_mape  inflation_only_mape
  stable           4.78             3.89                 3.56
volatile           1.02             2.31                 0.68
recovery           0.56             0.54                 0.53

Saved final_model_comparison.csv


In [38]:
print(f"\n{'='*70}")
print("SINGLE-VARIABLE TEST: ALL VARIABLES, ALL PERIODS")
print(f"{'='*70}")

all_single_var_results = []

for period_name in ["stable", "volatile", "recovery"]:
    period_df = df[df["period"] == period_name].sort_values("date").reset_index(drop=True)
    y = period_df["retail_sales"]

    for var in ["inflation_yoy", "interest_rate", "dollar_index"]:
        print(f"\n--- {period_name.upper()} — {var} only ---")

        X = period_df[[var]]
        order = pm.auto_arima(y, X=X, d=1, seasonal=False, trace=False, suppress_warnings=True, stepwise=True).order

        actuals, predictions = [], []

        for i in range(INITIAL_TRAIN_SIZE, len(period_df)):
            y_train = y.iloc[:i]
            X_train = X.iloc[:i]
            X_next = X.iloc[i:i+1]
            actual_value = y.iloc[i]

            model = ARIMA(y_train, order=order, exog=X_train)
            fitted_model = model.fit()
            forecast = fitted_model.forecast(steps=1, exog=X_next)
            predicted_value = forecast.iloc[0]

            actuals.append(actual_value)
            predictions.append(predicted_value)

        actuals = np.array(actuals)
        predictions = np.array(predictions)
        rmse = np.sqrt(np.mean((actuals - predictions) ** 2))
        mape = np.mean(np.abs((actuals - predictions) / actuals)) * 100

        print(f"{period_name} — {var}: RMSE = {rmse:,.2f}   MAPE = {mape:.2f}%")
        all_single_var_results.append({
            "period": period_name, "variable": var, "order": order, "rmse": rmse, "mape": round(mape, 2)
        })

print(f"\n{'='*70}")
print("FULL SINGLE-VARIABLE COMPARISON TABLE")
print(f"{'='*70}")
full_results_df = pd.DataFrame(all_single_var_results)
pivot_table = full_results_df.pivot(index="period", columns="variable", values="mape")
print(pivot_table.to_string())

full_results_df.to_csv("all_single_variable_results.csv", index=False)
pivot_table.to_csv("single_variable_mape_pivot.csv")
print("\nSaved all_single_variable_results.csv and single_variable_mape_pivot.csv")


SINGLE-VARIABLE TEST: ALL VARIABLES, ALL PERIODS

--- STABLE — inflation_yoy only ---
stable — inflation_yoy: RMSE = 28,598.16   MAPE = 3.56%

--- STABLE — interest_rate only ---
stable — interest_rate: RMSE = 32,548.80   MAPE = 4.42%

--- STABLE — dollar_index only ---
stable — dollar_index: RMSE = 31,273.24   MAPE = 4.53%

--- VOLATILE — inflation_yoy only ---


C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:989: EstimationWarning: Too few observations to estimate starting parameters for ARMA and trend. All parameters except for variances will be set to zeros.
  params_variance) = self._conditional_sum_squares(
C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:989: EstimationWarning: Too few observations to estimate starting parameters for ARMA and trend. All parameters except for variances will be set to zeros.
  params_variance) = self._conditional_sum_squares(
C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start

volatile — inflation_yoy: RMSE = 6,291.10   MAPE = 0.68%

--- VOLATILE — interest_rate only ---
volatile — interest_rate: RMSE = 21,397.60   MAPE = 1.97%

--- VOLATILE — dollar_index only ---


C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


volatile — dollar_index: RMSE = 10,729.94   MAPE = 1.35%

--- RECOVERY — inflation_yoy only ---


C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


recovery — inflation_yoy: RMSE = 4,357.20   MAPE = 0.53%

--- RECOVERY — interest_rate only ---
recovery — interest_rate: RMSE = 4,316.06   MAPE = 0.53%

--- RECOVERY — dollar_index only ---
recovery — dollar_index: RMSE = 4,511.10   MAPE = 0.56%

FULL SINGLE-VARIABLE COMPARISON TABLE
variable  dollar_index  inflation_yoy  interest_rate
period                                              
recovery          0.56           0.53           0.53
stable            4.53           3.56           4.42
volatile          1.35           0.68           1.97

Saved all_single_variable_results.csv and single_variable_mape_pivot.csv



## 7. VAR Model: Full Multivariate System

As a final test of model complexity, we fit a full Vector Autoregression (VAR) treating all four variables as jointly endogenous — first confirming stationarity of each.


In [39]:
from statsmodels.tsa.stattools import adfuller

print(f"\n{'='*70}")
print("ADF TEST: ALL 4 VARIABLES, ALL PERIODS")
print(f"{'='*70}")

variables = ["retail_sales", "inflation_yoy", "interest_rate", "dollar_index"]

for period_name in ["stable", "volatile", "recovery"]:
    print(f"\n--- {period_name.upper()} ---")
    period_df = df[df["period"] == period_name].sort_values("date")
    for var in variables:
        result = adfuller(period_df[var])
        verdict = "STATIONARY" if result[1] < 0.05 else "NON-STATIONARY (needs differencing)"
        print(f"  {var:15s} -> p-value = {result[1]:.4f} -> {verdict}")


ADF TEST: ALL 4 VARIABLES, ALL PERIODS

--- STABLE ---
  retail_sales    -> p-value = 0.0695 -> NON-STATIONARY (needs differencing)
  inflation_yoy   -> p-value = 0.0157 -> STATIONARY
  interest_rate   -> p-value = 0.8951 -> NON-STATIONARY (needs differencing)
  dollar_index    -> p-value = 0.9663 -> NON-STATIONARY (needs differencing)

--- VOLATILE ---
  retail_sales    -> p-value = 0.1831 -> NON-STATIONARY (needs differencing)
  inflation_yoy   -> p-value = 0.0100 -> STATIONARY
  interest_rate   -> p-value = 0.9968 -> NON-STATIONARY (needs differencing)
  dollar_index    -> p-value = 0.8432 -> NON-STATIONARY (needs differencing)

--- RECOVERY ---
  retail_sales    -> p-value = 1.0000 -> NON-STATIONARY (needs differencing)
  inflation_yoy   -> p-value = 0.3717 -> NON-STATIONARY (needs differencing)
  interest_rate   -> p-value = 0.5703 -> NON-STATIONARY (needs differencing)
  dollar_index    -> p-value = 0.9983 -> NON-STATIONARY (needs differencing)


C:\Users\HP\AppData\Local\Temp\ipykernel_24048\3290325758.py:13: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  result = adfuller(period_df[var])
C:\Users\HP\AppData\Local\Temp\ipykernel_24048\3290325758.py:13: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  result = adfuller(period_df[var])
C:\Users\HP\AppData\Local\Temp\ipykernel_24048\3290325758.py:13: FutureWarning: adfu

In [40]:
from statsmodels.tsa.api import VAR

print(f"\n{'='*70}")
print("VAR MODEL: ALL THREE PERIODS")
print(f"{'='*70}")

var_cols = ["retail_sales", "inflation_yoy", "interest_rate", "dollar_index"]

for period_name in ["stable", "volatile", "recovery"]:
    print(f"\n--- {period_name.upper()} ---")

    period_df = df[df["period"] == period_name].sort_values("date")[var_cols].reset_index(drop=True)

    # Difference all variables once, uniformly, for consistency
    period_diff = period_df.diff().dropna()

    # Fit VAR and let it select the best lag order using AIC
    model = VAR(period_diff)
    lag_selection = model.select_order(maxlags=4)
    best_lag = lag_selection.aic
    print(f"Best lag order (by AIC): {best_lag}")

    fitted = model.fit(best_lag if best_lag > 0 else 1)
    print(fitted.summary())


VAR MODEL: ALL THREE PERIODS

--- STABLE ---


C:\Users\HP\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use on the supported classes of index.
  self._init_dates(dates, freq)


ValueError: maxlags is too large for the number of observations and the number of equations. The largest model cannot be estimated.